# Agni — train the fire/smoke detector

Trains the detector the Agni rover runs, on a free Colab or Kaggle GPU instead of
your own machine.

Runtime → Change runtime type → **T4 GPU** before running anything. On CPU this
notebook will not finish.

At the end you download `best_ncnn_model`, copy it to the Pi, and start the rover
with `MODEL_PATH` pointing at it.


## 1. Check the GPU

If this prints nothing, you are on a CPU runtime. Stop and switch it.

In [ ]:
!nvidia-smi

## 2. Install

In [ ]:
!pip install -q ultralytics roboflow

import ultralytics
ultralytics.checks()

## 3. Get the dataset

Needs a free Roboflow key from https://app.roboflow.com/settings/api.

On Colab, put it in the Secrets panel (key icon, left sidebar) as `ROBOFLOW_API_KEY`.
On Kaggle, use Add-ons → Secrets. Pasting the key straight into a notebook you
later share leaks it.

In [ ]:
import os

try:  # Colab
    from google.colab import userdata
    api_key = userdata.get("ROBOFLOW_API_KEY")
except ImportError:  # Kaggle
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("ROBOFLOW_API_KEY")

from roboflow import Roboflow

project = Roboflow(api_key=api_key).workspace("sayed-gamall").project("fire-smoke-detection-yolov11")
dataset = project.version(2).download("yolov11")
print(dataset.location)

### Fix the dataset paths

Roboflow ships relative paths that assume its own working directory. Ultralytics
resolves them against a different root, so without this the run trains on zero
images and still reports success.

In [ ]:
import yaml
from pathlib import Path

config_path = Path(dataset.location) / "data.yaml"
config = yaml.safe_load(config_path.read_text())

for split, folder in (("train", "train"), ("val", "valid"), ("test", "test")):
    split_dir = Path(dataset.location) / folder / "images"
    if split_dir.is_dir():
        config[split] = str(split_dir)
        print(f"{split:<6} {len(list(split_dir.glob('*')))} images")
    else:
        config.pop(split, None)

config_path.write_text(yaml.safe_dump(config, default_flow_style=False))
print("classes:", config["names"])

## 4. Train

The reference recipe: yolo11n, 640px, 250 epochs, early stop after 20 epochs with
no improvement. A T4 fits `batch=16` comfortably; `batch=-1` sizes it automatically.

This takes hours. Colab disconnects idle sessions, so keep the tab open, and
re-run with `resume=True` if it drops.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
results = model.train(
    data=str(config_path),
    epochs=250,
    imgsz=640,
    batch=-1,
    patience=20,
    amp=True,
    plots=True,
    project="runs",
    name="agni-fire-smoke",
    exist_ok=True,
)
print(results.save_dir)

## 5. Check it before trusting it

This is a fire detector. Read recall on `Fire` specifically, not just the headline
mAP: a model that misses fire is worse than one that occasionally cries wolf.

In [ ]:
from pathlib import Path

best = Path(results.save_dir) / "weights" / "best.pt"
metrics = YOLO(str(best)).val(data=str(config_path), split="test")

for i, name in enumerate(config["names"]):
    print(f"{name:<6} precision={metrics.box.p[i]:.3f} recall={metrics.box.r[i]:.3f} mAP50={metrics.box.ap50[i]:.3f}")

In [ ]:
from IPython.display import Image, display

for plot in ("results.png", "confusion_matrix_normalized.png", "PR_curve.png"):
    path = Path(results.save_dir) / plot
    if path.exists():
        print(plot)
        display(Image(str(path)))

## 6. Export for the Raspberry Pi

NCNN, because a raw `.pt` runs PyTorch on the Pi's ARM CPU and is far slower at the
same accuracy. The input size is baked into the export, so whatever you set here
must match `DETECT_IMGSZ` on the rover.

In [ ]:
exported = YOLO(str(best)).export(format="ncnn", imgsz=640)
print(exported)

!zip -qr agni_ncnn_model.zip {exported}
print("zipped")

In [ ]:
try:
    from google.colab import files
    files.download("agni_ncnn_model.zip")
except ImportError:
    print("On Kaggle: find agni_ncnn_model.zip in the notebook output panel.")

## 7. Run it on the rover

Unzip on the Pi, then:

```bash
MODEL_PATH=/home/pi/best_ncnn_model DETECT_IMGSZ=640 DETECTOR=1   ROVER_TOKEN=yourtoken python3 rover_server.py
```

`MODEL_PATH` overrides the pretrained download, so the rover runs your weights and
never reaches the network for a model. If inference is too slow to drive against,
re-export at 416 and set `DETECT_IMGSZ=416` to match.
